# The Information Problem — Building a RAG Pipeline
## ME 493B — AI in Product Development | Mini-Project 2, Part A

**Instructor:** Scott Thielman, PhD — University of Washington Bothell
**Due:** Monday, April 27, 2026 at 11:59 PM
**Time estimate:** 60–90 minutes
**Points:** 50 (Part A). Part B is worth 50 points separately.

---

### The problem you're solving

In Sessions 5 and 6, you experienced the information problem firsthand. Your AI
assistant knew about aeroelastic coupling *in general* but not your client's specific
flutter margin data. It could discuss tolerance analysis *abstractly* but couldn't
tell you which tolerances your predecessor specified for the CardioSense enclosure.

**RAG (Retrieval-Augmented Generation)** solves this. Instead of hoping the model
"knows" your data, you *inject* the relevant documents into the prompt at query time.
In this notebook, you build a RAG pipeline from scratch.

### How this notebook works

Unlike MP1, **you write the code.** Each section describes what to build — the
concepts, the steps, the expected outputs. Use GitHub Copilot, Claude, ChatGPT, or
any AI coding tool to help. The learning is in understanding *what* you're building
and evaluating *whether* it works.

**Cell conventions:**
- Pre-written cells: Setup and data loading only (Section 0)
- Empty code cells: You implement, guided by the instructions above each cell
- **[X pts]** tags: Point value and expected output for grading

### Grading summary (50 pts)

| Section | Points | What the grader checks |
|---------|--------|----------------------|
| 1. Tokens & Context Windows | 8 | Token counts printed, budget calculation correct |
| 2. Embeddings for Retrieval | 8 | Ranked results with scores, answer extraction attempted |
| 3. Chunking Tradeoffs | 8 | Comparison table across 3 chunk sizes, best size identified |
| 4. RAG Pipeline | 10 | ChromaDB collection created, 5 queries run, accuracy reported |
| 5. Capstone (Attention paper) | 8 | 4 queries answered with retrieved evidence |
| 6. Reflections | 8 | Three thoughtful reflections (2–3 sentences each) |
| **Total** | **50** | |

### Building blocks from MP1

This notebook builds directly on three of the five building blocks you learned in MP1:

| Building Block | MP1 | MP2 Part A |
|---|---|---|
| **Representation** | Hand-crafted feature vectors, then learned embeddings | Embedding entire *documents* for retrieval |
| **Similarity** | Cosine similarity between individual vectors | Cosine similarity to rank documents against a query |
| **Search & Retrieval** | Mini search engine over 12 sentences | Full RAG pipeline over 20 documents + a research paper |

---
## Section 0: Setup

Run the cells below to install dependencies, import libraries, and load the
document corpus. **These are the only pre-written code cells in the notebook** —
everything else is yours to build.

The document corpus for this notebook lives in the `corpus/` folder alongside
this notebook — 20 text files from Ridgeline Engineering Partners, a fictional
mechanical engineering consultancy. A `manifest.json` file maps each filename to
its document ID, title, and type. The setup cell reads all of these automatically.

This is how real RAG systems work: your documents live as files on disk (or in a
database), and your pipeline loads and processes them programmatically.

In [ ]:
# ============================================================
# SETUP — Run this cell first (pre-written, do not modify)
# ============================================================

# Install additional packages (not in base requirements.txt)
import subprocess, sys
for pkg in ["tiktoken", "chromadb", "openai"]:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        stdout=subprocess.DEVNULL
    )

# Core imports
import numpy as np
import pandas as pd
import tiktoken
import chromadb
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import textwrap, os, time

# Load the embedding model (same one you used in MP1 Section 5)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Setup complete.")
print(f"  numpy {np.__version__}")
print(f"  pandas {pd.__version__}")
print(f"  tiktoken {tiktoken.__version__}")
print(f"  chromadb {chromadb.__version__}")
print(f"  Embedding model: all-MiniLM-L6-v2 (384-dim)")

In [ ]:
# Load document corpus from the corpus/ folder
# Each .txt file is one document; manifest.json maps filenames to metadata
import json, pathlib

corpus_dir = pathlib.Path("corpus")
if not corpus_dir.exists():
    # Handle running from repo root vs. from notebook directory
    corpus_dir = pathlib.Path("MP2/Part A/corpus")

with open(corpus_dir / "manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)

documents = []
for entry in manifest:
    filepath = corpus_dir / entry["filename"]
    text = filepath.read_text(encoding="utf-8")
    documents.append({
        "id": entry["id"],
        "title": entry["title"],
        "type": entry["type"],
        "text": text,
    })

print(f"Loaded {len(documents)} documents | Total characters: {sum(len(d['text']) for d in documents)}")
print(f"Source: {corpus_dir.resolve()}")
print()
for doc in documents[:5]:
    print(f"  {doc['id']}: {doc['title']} ({len(doc['text'])} chars)")
print(f"  ... and {len(documents) - 5} more")

In [ ]:
# "Attention Is All You Need" — abridged educational summary
# Vaswani et al., NeurIPS 2017. Full paper: https://arxiv.org/abs/1706.03762
attention_paper = """ATTENTION IS ALL YOU NEED — ABRIDGED EDUCATIONAL SUMMARY

The following is an abridged educational summary of: Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A.N., Kaiser, L., and Polosukhin, I. "Attention Is All You Need." Advances in Neural Information Processing Systems (NeurIPS), 2017. Read the full paper at https://arxiv.org/abs/1706.03762

This summary preserves key technical details and numerical results for educational use. It is not a verbatim reproduction of the original paper.

========================================
ABSTRACT
========================================

The dominant models for sequence-to-sequence tasks such as machine translation had traditionally been built on complex recurrent or convolutional neural networks arranged in an encoder-decoder configuration. The best-performing variants of these models also incorporated an attention mechanism to connect the encoder and decoder. This paper proposed a fundamentally new architecture called the Transformer, which is built solely on attention mechanisms and completely eliminates the need for recurrence and convolutions.

The authors demonstrated that this attention-only approach produces models that are not only higher quality but also far more parallelizable, requiring significantly less training time. On the WMT 2014 English-to-German translation benchmark, the Transformer achieved a BLEU score of 28.4, improving over the previous best results (including ensemble models) by more than 2 BLEU points. On the WMT 2014 English-to-French translation benchmark, the model established a new single-model state-of-the-art BLEU score of 41.8, after training for only 3.5 days on eight GPUs. This represented a small fraction of the training cost required by the previous best models. The authors further showed that the Transformer generalizes well to other tasks, applying it successfully to English constituency parsing with both large and limited training data.

========================================
INTRODUCTION AND MOTIVATION
========================================

Before the Transformer, recurrent neural networks — particularly Long Short-Term Memory networks (LSTMs) and Gated Recurrent Units (GRUs) — had been firmly established as the leading approaches for sequence modeling and transduction tasks like language modeling and machine translation. These architectures process sequences one element at a time: at each time step t, the model takes the current input and the previous hidden state h(t-1) to produce a new hidden state h(t). This sequential processing is fundamental to how recurrent networks work.

The problem with this sequential nature is that it prevents parallelization within individual training examples. Each step must wait for the previous step to complete before it can begin, because h(t) depends on h(t-1). As sequence lengths grow, this becomes a critical bottleneck. Memory constraints further limit how many sequences can be batched together during training. Researchers had made progress through factorization tricks and conditional computation to improve computational efficiency, but the fundamental constraint of sequential processing remained.

Attention mechanisms had already become an important part of sequence modeling, allowing models to capture dependencies between positions in a sequence regardless of their distance from each other. However, in nearly all prior work, attention was used alongside a recurrent network rather than as a replacement for it. The attention mechanism helped the recurrent model focus on relevant parts of the input, but the recurrent backbone still processed the sequence step by step.

The Transformer broke this paradigm entirely. Instead of using recurrence and adding attention on top, the authors proposed an architecture that relies entirely on attention mechanisms to model relationships between all positions in a sequence. By eliminating recurrence, the Transformer allows all positions in a sequence to be processed simultaneously, enabling massive parallelization during training. The paper demonstrated that this approach achieves superior translation quality while being trained in as little as twelve hours on eight NVIDIA P100 GPUs — dramatically faster than comparable recurrent models.

The key insight was that attention alone, without any recurrent or convolutional components, is sufficient to capture the relationships between words in a sequence. This was surprising because the conventional wisdom held that some form of sequential processing was necessary for understanding language, where word order and long-range dependencies are critical.

========================================
MODEL ARCHITECTURE OVERVIEW
========================================

The Transformer uses an encoder-decoder structure, which is common in sequence-to-sequence models. The encoder reads the input sequence (for example, a sentence in English) and produces a continuous representation of it. The decoder then generates the output sequence (for example, the translation in German) one element at a time, using the encoder's representation along with the previously generated output elements.

What makes the Transformer unique is how the encoder and decoder are built. Instead of recurrent layers, both use stacked layers of self-attention and feed-forward networks.

The encoder consists of a stack of N = 6 identical layers. Each layer contains two sub-layers. The first sub-layer is a multi-head self-attention mechanism, which allows each position in the sequence to attend to all other positions. The second sub-layer is a simple position-wise fully connected feed-forward network, which processes each position independently. Around each sub-layer, the model uses a residual connection (adding the sub-layer's input to its output) followed by layer normalization. The formula for this is: LayerNorm(x + Sublayer(x)). All sub-layers and embedding layers produce outputs of dimension d_model = 512, which ensures that residual connections can work (the dimensions must match for addition).

The decoder is also composed of a stack of N = 6 identical layers. It has the same two sub-layers as the encoder (self-attention and feed-forward), but adds a third sub-layer that performs multi-head attention over the encoder's output. This encoder-decoder attention allows each position in the decoder to look at all positions in the input sequence. The decoder's self-attention sub-layer is modified with masking to prevent positions from attending to future positions. This is essential because during generation, the model predicts one token at a time and should not have access to tokens that haven't been generated yet. The masking, combined with the output embeddings being offset by one position, ensures that the prediction for position i depends only on known outputs at positions less than i.

========================================
SCALED DOT-PRODUCT ATTENTION
========================================

The attention mechanism at the heart of the Transformer can be understood through three key concepts: queries (Q), keys (K), and values (V). An attention function maps a query and a set of key-value pairs to an output. The output is a weighted sum of the values, where the weight for each value is determined by a compatibility function between the query and the corresponding key.

Think of it like a search engine: the query is your search term, the keys are the index entries of all the documents, and the values are the actual document contents. The attention mechanism scores how well each key matches the query, then returns a weighted combination of the corresponding values — paying more attention to the values whose keys best match the query.

The specific form used in the Transformer is called Scaled Dot-Product Attention. Given queries and keys of dimension d_k and values of dimension d_v, the attention is computed as:

    Attention(Q, K, V) = softmax(Q * K^T / sqrt(d_k)) * V

Here is what each step does:
1. Compute Q * K^T: the dot product of queries with all keys gives a score for each query-key pair, measuring compatibility.
2. Divide by sqrt(d_k): this scaling factor prevents the dot products from growing too large in magnitude. Without scaling, when d_k is large, the dot products become very large, pushing the softmax function into regions where its gradients are extremely small (effectively saturating it). This would make learning very slow.
3. Apply softmax: convert the scaled scores into probabilities that sum to 1. Higher scores get exponentially more weight.
4. Multiply by V: produce the output as a weighted sum of the value vectors.

The authors chose dot-product attention over additive attention (which uses a small neural network to compute compatibility scores) because dot-product attention is much faster and more space-efficient in practice. It can be implemented using highly optimized matrix multiplication routines. While additive attention performs comparably for small key dimensions, scaled dot-product attention wins for the larger dimensions used in practice.

In implementation, the authors compute attention for all queries simultaneously by packing them into a matrix Q, with keys and values similarly packed into matrices K and V. This allows the entire attention computation to be expressed as matrix operations, which modern GPUs handle extremely efficiently.

========================================
MULTI-HEAD ATTENTION
========================================

Rather than computing a single attention function using the full d_model = 512 dimensional keys, values, and queries, the authors found it beneficial to split the computation into multiple parallel attention "heads." This is multi-head attention.

The idea is straightforward: instead of one attention computation, perform h = 8 parallel attention computations, each operating on a different learned linear projection of the queries, keys, and values. Specifically:
- Project Q, K, and V down from 512 dimensions to d_k = d_v = d_model/h = 64 dimensions using learned weight matrices
- Compute attention independently for each of the 8 heads
- Concatenate the 8 outputs (each 64-dimensional, giving 512 total)
- Apply a final linear projection to produce the output

The mathematical formulation is:
    MultiHead(Q, K, V) = Concat(head_1, ..., head_h) * W^O
    where head_i = Attention(Q * W_i^Q, K * W_i^K, V * W_i^V)

The key advantage of multi-head attention is that it allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, the output at each position is an average over all attended positions, which tends to inhibit the model's ability to focus on multiple relevant pieces of information simultaneously. For example, when processing the word "it" in a sentence, one head might attend to the noun that "it" refers to, while another head attends to the verb that "it" is the subject of.

Because each head operates on reduced dimensionality (64 instead of 512), the total computational cost of multi-head attention is similar to that of single-head attention with the full dimensionality. The parallelism across heads is free from a compute perspective.

The Transformer uses multi-head attention in three different ways:
1. Encoder self-attention: In the encoder, queries, keys, and values all come from the output of the previous encoder layer. Each position in the encoder can attend to all positions in the previous layer, allowing the model to capture relationships between any pair of words in the input sentence.
2. Decoder self-attention (masked): In the decoder, self-attention works similarly, but with masking that prevents each position from attending to subsequent positions. This preserves the autoregressive property needed for generation.
3. Encoder-decoder attention: In the decoder, queries come from the previous decoder layer while keys and values come from the encoder output. This allows every position in the decoder to attend over all positions in the input sequence, similar to the attention mechanism used in traditional encoder-decoder architectures.

========================================
POSITION-WISE FEED-FORWARD NETWORKS
========================================

In addition to attention, each layer in both the encoder and decoder contains a position-wise fully connected feed-forward network. This network is applied independently and identically to each position in the sequence. It consists of two linear transformations with a ReLU activation in between:

    FFN(x) = max(0, x * W_1 + b_1) * W_2 + b_2

The inner layer has dimensionality d_ff = 2048, while the input and output dimensions are d_model = 512. Although the same function is applied at every position, the parameters (W_1, W_2, b_1, b_2) differ from layer to layer. This can also be described as two convolutions with kernel size 1. The feed-forward network gives the model the capacity to perform non-linear transformations on each position's representation independently, complementing the attention mechanism which mixes information across positions.

========================================
POSITIONAL ENCODING
========================================

Because the Transformer contains no recurrence and no convolution, it has no inherent mechanism for understanding the order of elements in a sequence. The word "dog bites man" and "man bites dog" would produce identical representations without some way to encode position information. To address this, the authors inject information about the position of each token in the sequence by adding positional encodings to the input embeddings at the bottom of both the encoder and decoder stacks.

The positional encodings have the same dimension d_model = 512 as the embeddings, so that the two can be added together (element-wise sum). The authors used sinusoidal functions of different frequencies to generate the positional encodings:

    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

where pos is the position in the sequence and i is the dimension index. Each dimension of the positional encoding corresponds to a sinusoid with a different wavelength, forming a geometric progression from 2*pi to 10000*2*pi.

The authors chose sinusoidal functions because they hypothesized that this representation would allow the model to learn to attend by relative positions. For any fixed offset k, the positional encoding PE(pos+k) can be expressed as a linear function of PE(pos). This means the model can potentially learn to compute "the word 3 positions to the left" as a simple linear operation on the positional encodings.

The authors also experimented with using learned positional embeddings (trainable vectors, one for each position) instead of the fixed sinusoidal encodings. The two approaches produced nearly identical results. They chose the sinusoidal version because it has the potential to extrapolate to sequence lengths longer than any seen during training, since the mathematical function is defined for any position value.

========================================
WHY SELF-ATTENTION
========================================

A natural question is: why use self-attention instead of recurrent or convolutional layers? The authors compared these approaches along three dimensions:

1. Computational complexity per layer: How much total computation does each layer require?
2. Parallelizable computation: How much of the computation can be done in parallel (measured as the minimum number of sequential operations)?
3. Maximum path length: What is the longest path a signal must travel between any two positions in the sequence? Shorter paths between any pair of positions make it easier to learn long-range dependencies.

The comparison (for a sequence of length n and representation dimension d):

Self-Attention:
- Complexity per layer: O(n^2 * d)
- Sequential operations: O(1) — all positions attend to all other positions in parallel
- Maximum path length: O(1) — any position can directly attend to any other position

Recurrent layers:
- Complexity per layer: O(n * d^2)
- Sequential operations: O(n) — must process the sequence step by step
- Maximum path length: O(n) — information from position 1 must pass through all intermediate hidden states to reach position n

Convolutional layers:
- Complexity per layer: O(k * n * d^2) where k is kernel size
- Sequential operations: O(1)
- Maximum path length: O(log_k(n)) — requires a stack of layers to connect distant positions

Self-attention connects every pair of positions in a single operation, giving a maximum path length of O(1). Recurrent layers require O(n) sequential steps for information to travel between distant positions. Convolutional layers need O(log_k(n)) stacked layers to achieve full connectivity. This short path length is a key advantage of self-attention for learning long-range dependencies.

In terms of raw computation, self-attention is faster than recurrence when the sequence length n is smaller than the representation dimension d. This is the typical case for most sentence-level tasks in machine translation, where sequences might be 50-100 tokens long but representations are 512 dimensions. For very long sequences, the n^2 factor in self-attention becomes expensive, and the authors suggested a restricted variant where each position attends only to a local neighborhood of size r.

An additional benefit noted by the authors is that self-attention produces more interpretable models. When examining the attention distributions, individual attention heads clearly learn to perform different tasks. Some heads track syntactic structure, others handle coreference resolution (determining what "it" refers to), and others capture long-distance dependencies between verbs and their arguments.

========================================
TRAINING DETAILS
========================================

The Transformer was trained on standard machine translation benchmarks:

For English-to-German translation: the WMT 2014 dataset containing approximately 4.5 million sentence pairs. Sentences were encoded using byte-pair encoding with a shared source-target vocabulary of approximately 37,000 tokens.

For English-to-French translation: the significantly larger WMT 2014 dataset containing approximately 36 million sentence pairs, with a vocabulary of 32,000 tokens using word-piece encoding.

Training was performed on a machine with 8 NVIDIA P100 GPUs. The base model was trained for approximately 100,000 steps (about 12 hours), with each training step processing batches of sentence pairs containing approximately 25,000 source tokens and 25,000 target tokens. The larger "big" model was trained for 300,000 steps (approximately 3.5 days).

The optimizer was Adam with beta_1 = 0.9, beta_2 = 0.98, and epsilon = 10^-9. A custom learning rate schedule was used that increases the learning rate linearly during a warmup period (4,000 steps for the base model), then decreases it proportionally to the inverse square root of the step number. This warmup prevents large updates early in training when the model parameters are still random.

Two forms of regularization were applied:
1. Residual dropout: Applied to the output of each sub-layer before it is added to the sub-layer input and normalized. Also applied to the sums of the embeddings and positional encodings. The dropout rate was P_drop = 0.1 for the base model and 0.3 for the big model.
2. Label smoothing: During training, label smoothing with value epsilon_ls = 0.1 was used. This technique replaces hard target labels (0 or 1) with smoothed values (e.g., 0.9 for the correct class, distributing 0.1 across other classes). While this hurts perplexity (the model becomes less confident in its predictions), it improves accuracy and BLEU scores because it encourages the model to be less overconfident.

========================================
RESULTS
========================================

Machine Translation Results:

On the WMT 2014 English-to-German translation task, the big Transformer model achieved a BLEU score of 28.4. This surpassed all previously reported models, including ensembles of multiple models, by more than 2.0 BLEU points. This was a new state-of-the-art result. Even the smaller base Transformer model achieved 27.3 BLEU, outperforming all previously published single models and ensembles at a tiny fraction of their training cost.

On the WMT 2014 English-to-French translation task, the big Transformer achieved a BLEU score of 41.8, establishing a new single-model state-of-the-art. This was accomplished at less than one-quarter of the training cost of the previous best model. The base model achieved 38.1 BLEU on this task. For context, the previous state-of-the-art single models scored around 40.5 BLEU and required far more computational resources.

The training cost comparison is striking. The Transformer (big) required approximately 2.3 x 10^19 FLOPs for English-to-German, compared to 1.8 x 10^20 FLOPs for the best previous ensemble model — nearly an order of magnitude less computation for a better result.

Model Architecture Variation Experiments:

The authors conducted extensive experiments varying different aspects of the Transformer architecture to understand what matters:

Attention heads: Varying the number of attention heads while keeping total computation constant revealed that single-head attention is 0.9 BLEU worse than the best setting (8 heads). However, too many heads (32 heads with dimension 16 each) also degraded quality slightly. The sweet spot was around 8 heads with 64 dimensions each.

Attention key dimension: Reducing the attention key dimension d_k hurt model quality, suggesting that the compatibility function between queries and keys benefits from higher-dimensional representations. A more sophisticated compatibility function than simple dot product might help with smaller key dimensions.

Model size: As expected, larger models (more dimensions, more layers) performed better, up to the limits of what could be trained on the available hardware.

Dropout: Dropout was critical for avoiding overfitting. Without dropout, performance degraded significantly. A dropout rate of 0.1 worked well for the base model.

Positional encoding: Replacing sinusoidal positional encodings with learned positional embeddings produced nearly identical results, suggesting that the model is relatively insensitive to the exact form of positional information.

English Constituency Parsing:

To demonstrate that the Transformer generalizes beyond translation, the authors applied it to English constituency parsing (analyzing grammatical structure). With a 4-layer Transformer trained only on the Wall Street Journal portion of the Penn Treebank (about 40,000 sentences), the model achieved 91.3 F1 on the WSJ test set — better than all previous discriminative models except one, and without any task-specific architectural modifications. In a semi-supervised setting with additional data, the model reached 92.7 F1.

========================================
CONCLUSION
========================================

The Transformer was the first sequence transduction model built entirely on attention, replacing the recurrent layers that had been standard in encoder-decoder architectures with multi-headed self-attention. This design choice had profound consequences:

Training speed: The Transformer can be trained significantly faster than architectures based on recurrent or convolutional layers, because self-attention allows all positions to be processed in parallel. The 3.5-day training time for state-of-the-art translation results was dramatically less than previous approaches.

Translation quality: The model established new state-of-the-art results on both the English-to-German and English-to-French WMT 2014 benchmarks, with the English-to-German result surpassing even ensemble models despite using far less computation.

Generalizability: The architecture transferred successfully to constituency parsing without modification, suggesting that attention-based models are broadly applicable across sequence tasks.

The authors outlined several directions for future work: applying the Transformer to problems involving input and output modalities other than text (such as images, audio, and video), and investigating ways to make the generation process less sequential. They also suggested exploring local, restricted attention mechanisms for handling very long sequences efficiently.

Historical significance: Although the authors could not have known it at the time, the Transformer architecture became the foundation for virtually all subsequent breakthroughs in natural language processing. BERT (2018) used the Transformer encoder for bidirectional language understanding. GPT (2018) and its successors (GPT-2, GPT-3, GPT-4) used the Transformer decoder for autoregressive text generation. Every major large language model today — including Claude, GPT-4, Gemini, and Llama — is built on the Transformer architecture introduced in this paper. The attention mechanism described here is also the foundation of retrieval systems, including the RAG pipeline you are building in this notebook.
"""

print(f"Paper loaded: {len(attention_paper)} characters")

---
## Section 1: Tokens and Context Windows [8 pts]

Before you can manage context, you need to understand the **budget**. Every language
model has a finite context window measured in tokens — not words, not characters,
but *tokens*. Your job as an engineer using AI is to ensure the right information
fits within that budget.

### What are tokens?

A token is the fundamental unit a language model reads and generates. Most modern
models use **subword tokenization** — common words like "the" or "engineering" map
to a single token, while rare or technical words get split into pieces. For example,
"aeroelastic" might become ["aero", "elastic"], using two tokens. The tokenizer
`cl100k_base` is used by GPT-4-class models (and is representative of how
Claude-class models tokenize as well).

Different models use different tokenizers, so the same text produces different token
counts depending on which model you're targeting. The context window is a **hard
budget** — not a suggestion. If your prompt exceeds it, the API returns an error.

### What to build

1. Create a `tiktoken` encoder using the `cl100k_base` encoding
2. Tokenize one of the Ridgeline documents — print the raw text, the token list
   (first 20 tokens), and the total token count
3. Tokenize **all 20 documents**. Build and print a summary table (DataFrame):
   document ID, title, character count, token count
4. Calculate: if the context window is 8,192 tokens and you reserve 1,000 for the
   question + answer, how many documents can you fit in the remaining 7,192 tokens?
   Iterate through documents sorted by token count (smallest first) and pack as many
   as fit. Print the count and remaining tokens.

### Key concepts

- Tokens are not words — subword tokenization splits rare words and joins common ones
- The ratio of characters-to-tokens is typically 3–5 for English text
- Context window is a hard constraint, not a suggestion
- **This is why we need retrieval** — you can't paste all 20 documents into a prompt

### Reference links

- [Tiktokenizer](https://tiktokenizer.vercel.app/) — paste text and see tokens interactively
- [Anthropic Context Windows](https://docs.anthropic.com/en/docs/about-claude/models) — see how context windows vary by model

### What to submit [8 pts]

Print these exact outputs (the grader checks for them):

```
Document DOC-001: {N} characters → {M} tokens (ratio: {N/M:.1f} chars/token)
```

A summary table of all 20 documents with columns: ID, Title, Characters, Tokens

```
At 8,192 token context window with 1,000 reserved: {X} documents fit fully, {Y} tokens remaining
```

> **Forward connection:** You can't paste all 20 documents into a prompt. That's why
> we need retrieval — finding the RIGHT documents for each question.

In [ ]:
# Section 1, Steps 1-2: Tokenize DOC-001

# Step 1: Create a tiktoken encoder using cl100k_base (GPT-4 family tokenizer)
encoder = tiktoken.get_encoding("cl100k_base")

# Step 2: Tokenize DOC-001
doc_001 = documents[0]
doc_001_tokens = encoder.encode(doc_001["text"])

print(f"--- DOC-001: {doc_001['title']} ---")
print(f"Raw text (first 300 chars):\n{doc_001['text'][:300]}...\n")
print(f"First 20 tokens (as integers): {doc_001_tokens[:20]}")
print(f"First 20 tokens (decoded):")
for t in doc_001_tokens[:20]:
    print(f"  {t} → '{encoder.decode([t])}'")
print()

N = len(doc_001["text"])
M = len(doc_001_tokens)
print(f"Document DOC-001: {N} characters → {M} tokens (ratio: {N/M:.1f} chars/token)")

In [ ]:
# Section 1, Step 3: Tokenize all 20 documents and build summary table

rows = []
for doc in documents:
    tokens = encoder.encode(doc["text"])
    rows.append({
        "ID": doc["id"],
        "Title": doc["title"],
        "Characters": len(doc["text"]),
        "Tokens": len(tokens),
    })

token_df = pd.DataFrame(rows)
print(token_df.to_string(index=False))
print(f"\nTotal: {token_df['Characters'].sum()} characters, {token_df['Tokens'].sum()} tokens")

In [ ]:
# Section 1, Step 4: Context window budget calculation

context_window = 8192
reserved = 1000
budget = context_window - reserved  # 7,192 tokens available

# Sort documents by token count (smallest first) and pack greedily
sorted_docs = token_df.sort_values("Tokens").reset_index(drop=True)

tokens_used = 0
docs_fit = 0
for _, row in sorted_docs.iterrows():
    if tokens_used + row["Tokens"] <= budget:
        tokens_used += row["Tokens"]
        docs_fit += 1
    else:
        break

remaining = budget - tokens_used
print(f"At 8,192 token context window with 1,000 reserved: {docs_fit} documents fit fully, {remaining} tokens remaining")

---
## Section 2: Embeddings for Document Retrieval [8 pts]

You used embeddings in MP1 to measure similarity between individual sentences. Now
apply the same idea at document scale: embed each document, embed a query, and find
the most relevant documents by cosine similarity.

The model is the same `all-MiniLM-L6-v2` you used in MP1 Section 5 — it maps any
text to a 384-dimensional vector. Cosine similarity (MP1 Section 2) measures how
closely two vectors point in the same direction, regardless of magnitude.

### What to build

1. Embed all 20 document texts using `embedding_model.encode()` — you'll get a
   (20, 384) matrix
2. Embed the query: `"What is the standard billing rate for senior engineers?"`
3. Compute cosine similarity between the query embedding and all 20 document embeddings
   (use `sklearn.metrics.pairwise.cosine_similarity`)
4. Print a ranked list: top 5 documents by similarity, with their scores
5. Read the top-ranked document text — does it contain the answer? Print what you find.
6. Run a second, harder query: `"What rating factor was found for the interior girders on the bridge project?"`
   Repeat steps 3–5 for this query.

### Key concepts

- This is the same cosine similarity from MP1 Section 2, now applied to real documents
- Embedding quality depends on the model — `all-MiniLM-L6-v2` was trained on semantic
  similarity tasks (sentence pairs), so it captures meaning, not just keywords
- Whole-document embeddings work for short documents but lose detail for long ones —
  a 500-word document about five different topics gets one "averaged" embedding that
  may not match any specific fact well. This motivates chunking (Section 3).

### Reference links

- [Sentence-Transformers documentation](https://www.sbert.net/)
- [MTEB Benchmark Leaderboard](https://huggingface.co/spaces/mteb/leaderboard) — compare embedding models

### What to submit [8 pts]

For **each** of the two queries, print:

```
Query: '{query text}'
Top match: DOC-{XXX} '{title}' (similarity: {score:.4f})
```

Ranked list of top 5 documents with similarity scores.

For the top match, print either:
```
Answer found in document: {the specific answer}
```
or:
```
Answer NOT found in top document — [your explanation of why]
```

In [ ]:
# Section 2, Steps 1-5: Embed all documents and run first query

# Step 1: Embed all 20 document texts → (20, 384) matrix
doc_texts = [doc["text"] for doc in documents]
doc_embeddings = embedding_model.encode(doc_texts)
print(f"Document embeddings shape: {doc_embeddings.shape}")

# Step 2: Embed the query
query_1 = "What is the standard billing rate for senior engineers?"
query_1_embedding = embedding_model.encode([query_1])

# Step 3: Compute cosine similarity between query and all documents
similarities_1 = cosine_similarity(query_1_embedding, doc_embeddings)[0]

# Step 4: Rank and print top 5
ranked_indices_1 = np.argsort(similarities_1)[::-1]
print(f"\nQuery: '{query_1}'")
print(f"Top match: {documents[ranked_indices_1[0]]['id']} '{documents[ranked_indices_1[0]]['title']}' (similarity: {similarities_1[ranked_indices_1[0]]:.4f})")
print("\nTop 5 documents by similarity:")
for rank, idx in enumerate(ranked_indices_1[:5], 1):
    print(f"  {rank}. {documents[idx]['id']}: {documents[idx]['title']} (similarity: {similarities_1[idx]:.4f})")

# Step 5: Read the top-ranked document — does it contain the answer?
top_doc_1 = documents[ranked_indices_1[0]]
print(f"\n--- Top document text (first 500 chars) ---")
print(top_doc_1["text"][:500])
# Check for the answer
if "senior engineer" in top_doc_1["text"].lower() or "billing rate" in top_doc_1["text"].lower():
    # Extract relevant line
    for line in top_doc_1["text"].split("\n"):
        if "senior engineer" in line.lower() or "Senior Engineer" in line:
            print(f"\nAnswer found in document: {line.strip()}")
            break
else:
    print("\nAnswer NOT found in top document — the document may not contain billing rate information.")

In [ ]:
# Section 2, Step 6: Second query (harder)

query_2 = "What rating factor was found for the interior girders on the bridge project?"
query_2_embedding = embedding_model.encode([query_2])

# Compute cosine similarity
similarities_2 = cosine_similarity(query_2_embedding, doc_embeddings)[0]

# Rank and print top 5
ranked_indices_2 = np.argsort(similarities_2)[::-1]
print(f"Query: '{query_2}'")
print(f"Top match: {documents[ranked_indices_2[0]]['id']} '{documents[ranked_indices_2[0]]['title']}' (similarity: {similarities_2[ranked_indices_2[0]]:.4f})")
print("\nTop 5 documents by similarity:")
for rank, idx in enumerate(ranked_indices_2[:5], 1):
    print(f"  {rank}. {documents[idx]['id']}: {documents[idx]['title']} (similarity: {similarities_2[idx]:.4f})")

# Read the top-ranked document — does it contain the answer?
top_doc_2 = documents[ranked_indices_2[0]]
print(f"\n--- Top document text (first 500 chars) ---")
print(top_doc_2["text"][:500])

# Try to extract the answer
text_lower = top_doc_2["text"].lower()
if "rating factor" in text_lower and "interior" in text_lower:
    for line in top_doc_2["text"].split("\n"):
        if "interior" in line.lower() and ("rating" in line.lower() or "girder" in line.lower()):
            print(f"\nAnswer found in document: {line.strip()}")
            break
    else:
        for line in top_doc_2["text"].split("\n"):
            if "rating factor" in line.lower():
                print(f"\nAnswer found in document: {line.strip()}")
                break
else:
    print("\nAnswer NOT found in top document — whole-document embeddings average over "
          "many topics, so a specific fact like a rating factor for interior girders may "
          "be diluted by surrounding text. This motivates chunking (Section 3).")

---
## Section 3: Chunking and Its Consequences [8 pts]

Whole-document embeddings lose specificity for long documents. A document covering
five topics gets one "averaged" vector that poorly represents any individual fact.
**Chunking** breaks documents into smaller pieces so each embedding is more focused.

But chunk size is an engineering tradeoff — exactly like mesh density in FEA
(Session 6, slide 13):

| Chunk too small | Chunk too large |
|---|---|
| Lose context between related sentences | Dilute specific facts with surrounding noise |
| A chunk saying "0.73" without "rating factor" or "interior girders" is useless | Similarity score drops because the embedding averages over irrelevant text |

**Overlap** helps: facts that span a chunk boundary are captured in both the
overlapping chunks.

### What to build

1. Write a chunking function: `chunk_text(text, chunk_size, overlap)` that splits
   text into chunks of `chunk_size` characters with `overlap` characters of overlap.
   Each chunk should also carry metadata about its source document.
2. Chunk all 20 documents at three sizes: **200**, **500**, and **1000** characters
   (use overlap = 20% of chunk_size: 40, 100, 200 respectively)
3. For each chunk size, print: total chunks created, average chunk length in
   characters, smallest chunk, largest chunk
4. Re-embed ALL chunks at each size and re-run the two queries from Section 2
5. Compare: which chunk size retrieves the most relevant chunk? Build a comparison
   table showing chunk_size | num_chunks | query_1_top_score | query_2_top_score

### Key concepts

- Chunking is an engineering tradeoff — not a parameter you optimize once and forget
- Smaller chunks = more precise retrieval but risk losing context
- Larger chunks = more context per result but diluted similarity scores
- Overlap prevents losing facts that fall on chunk boundaries
- The "right" chunk size depends on your documents and your questions

### Reference links

- [LangChain Text Splitters](https://docs.langchain.com/oss/javascript/integrations/splitters/index) — chunking strategies (JS examples, concepts are universal)
- [Greg Kamradt's Chunking Visualization](https://github.com/FullStackRetrieval-com/RetrievalTutorials) — visual intuition

### What to submit [8 pts]

Comparison table:

```
chunk_size | num_chunks | query_1_top_score | query_2_top_score
200        | ...        | ...               | ...
500        | ...        | ...               | ...
1000       | ...        | ...               | ...
```

```
Best chunk size for query 1: {size} chars (score: {score:.4f})
Best chunk size for query 2: {size} chars (score: {score:.4f})
```

Print the actual text of the top-retrieved chunk for each query at the best chunk
size — visually confirm it contains the answer.

In [ ]:
# Section 3, Step 1: Implement the chunking function

def chunk_text(text, chunk_size, overlap):
    """Split text into chunks of chunk_size characters with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# Quick test
test_chunks = chunk_text("ABCDEFGHIJKLMNOPQRSTUVWXYZ", chunk_size=10, overlap=3)
print(f"Test: 26-char string → {len(test_chunks)} chunks of size 10, overlap 3:")
for i, c in enumerate(test_chunks):
    print(f"  Chunk {i}: '{c}' ({len(c)} chars)")

In [ ]:
# Section 3, Step 2-3: Chunk all 20 documents at 3 sizes

chunk_configs = [
    {"chunk_size": 200, "overlap": 40},
    {"chunk_size": 500, "overlap": 100},
    {"chunk_size": 1000, "overlap": 200},
]

# Store chunks for each config for later embedding
all_chunked = {}

for config in chunk_configs:
    cs = config["chunk_size"]
    ov = config["overlap"]
    chunks_with_meta = []
    for doc in documents:
        doc_chunks = chunk_text(doc["text"], cs, ov)
        for i, chunk in enumerate(doc_chunks):
            chunks_with_meta.append({
                "text": chunk,
                "source_doc_id": doc["id"],
                "title": doc["title"],
                "chunk_index": i,
            })
    all_chunked[cs] = chunks_with_meta
    lengths = [len(c["text"]) for c in chunks_with_meta]
    print(f"Chunk size {cs} (overlap {ov}):")
    print(f"  Total chunks: {len(chunks_with_meta)}")
    print(f"  Avg length: {np.mean(lengths):.1f} chars")
    print(f"  Min length: {min(lengths)} chars")
    print(f"  Max length: {max(lengths)} chars")
    print()

In [ ]:
# Section 3, Steps 4-5: Embed all chunks, re-run queries, build comparison table

query_1 = "What is the standard billing rate for senior engineers?"
query_2 = "What rating factor was found for the interior girders on the bridge project?"

q1_emb = embedding_model.encode([query_1])
q2_emb = embedding_model.encode([query_2])

comparison_rows = []
best_chunks = {}  # Store best chunk text for each query

for cs in [200, 500, 1000]:
    chunks = all_chunked[cs]
    chunk_texts = [c["text"] for c in chunks]

    # Embed all chunks at this size
    chunk_embs = embedding_model.encode(chunk_texts, show_progress_bar=False)

    # Query 1
    sims_1 = cosine_similarity(q1_emb, chunk_embs)[0]
    top_idx_1 = np.argmax(sims_1)
    top_score_1 = sims_1[top_idx_1]

    # Query 2
    sims_2 = cosine_similarity(q2_emb, chunk_embs)[0]
    top_idx_2 = np.argmax(sims_2)
    top_score_2 = sims_2[top_idx_2]

    comparison_rows.append({
        "chunk_size": cs,
        "num_chunks": len(chunks),
        "query_1_top_score": round(top_score_1, 4),
        "query_2_top_score": round(top_score_2, 4),
    })

    # Track best chunks
    if "q1" not in best_chunks or top_score_1 > best_chunks["q1"]["score"]:
        best_chunks["q1"] = {"size": cs, "score": top_score_1, "text": chunks[top_idx_1]["text"], "source": chunks[top_idx_1]["source_doc_id"]}
    if "q2" not in best_chunks or top_score_2 > best_chunks["q2"]["score"]:
        best_chunks["q2"] = {"size": cs, "score": top_score_2, "text": chunks[top_idx_2]["text"], "source": chunks[top_idx_2]["source_doc_id"]}

# Print comparison table
comp_df = pd.DataFrame(comparison_rows)
print("Comparison table:")
print(comp_df.to_string(index=False))

print(f"\nBest chunk size for query 1: {best_chunks['q1']['size']} chars (score: {best_chunks['q1']['score']:.4f})")
print(f"Best chunk size for query 2: {best_chunks['q2']['size']} chars (score: {best_chunks['q2']['score']:.4f})")

print(f"\n--- Best chunk for Query 1 [{best_chunks['q1']['source']}] ---")
print(best_chunks["q1"]["text"])

print(f"\n--- Best chunk for Query 2 [{best_chunks['q2']['source']}] ---")
print(best_chunks["q2"]["text"])

---
## Section 4: Building the RAG Pipeline [10 pts]

Now assemble the full pipeline: chunk documents → embed chunks → store in a vector
database → query → retrieve relevant chunks → generate a grounded answer. This is
the machinery behind every AI system that uses your data.

### The RAG stack

| Component | Tool | Why |
|-----------|------|-----|
| Embedding | `all-MiniLM-L6-v2` | Local, fast, no API key needed |
| Vector store | ChromaDB | Handles storage + similarity search — you don't compute cosine manually against every chunk |
| Generation | GitHub Models API (`openai` SDK) | Free for GitHub accounts, OpenAI-compatible |

**ChromaDB** is the vector store from the RAG pipeline diagram in Session 6 (slide 12):
documents → chunk & embed → **vector store** → query → retrieve → generate.
It handles embedding storage and nearest-neighbor search efficiently.

**GitHub Models API** provides free access to models like GPT-4o via the `openai`
Python SDK — same SDK, just a different base URL. You'll use your existing GitHub
Personal Access Token (PAT).

### Setting up your GitHub PAT

You need a **fine-grained** GitHub Personal Access Token with Models permission:

1. Go to [github.com/settings/tokens?type=beta](https://github.com/settings/tokens?type=beta)
2. Click **"Generate new token"** (this page defaults to fine-grained tokens —
   do **not** use classic tokens)
3. Give it a name (e.g., "ME493B Models") and set expiration
4. Under **Permissions → Account permissions → Models**, select **"Read"**
5. Click "Generate token" and **copy the token immediately** — you won't see
   it again

### Storing your token in a `.env` file

Your token goes in a `.env` file in the root of your repository. This file is
already listed in `.gitignore`, so it won't be committed or shared.

**If you don't have a `.env` file yet:** ask Copilot! Open Copilot Chat and type:
*"Create a .env file in my repo root with a GITHUB_TOKEN variable."* It will
generate the file for you — just paste in your actual token value.

The `.env` file should look like this:
```
GITHUB_TOKEN=github_pat_your_token_here
```

The setup cell below loads the token from `.env` automatically.

**Rate limits:** GitHub Models free tier is rate-limited (~10 requests/min for
GPT-4o, ~50/day). This is enough for the exercises. If you hit limits, wait a minute
or switch to another model (e.g., `openai/gpt-4o-mini`).

### What to build

1. Choose a chunk size based on your Section 3 results — justify your choice in a
   comment
2. Chunk all documents at that size
3. Create a ChromaDB collection and add all chunks with embeddings and metadata
   (source document ID, title, chunk index)
4. Write `query_rag(question, n_results=3)`: embed the question, query ChromaDB,
   return top-N chunks with metadata
5. Write `generate_answer(question, retrieved_chunks)`: format chunks into a prompt,
   call GitHub Models API, return the answer. The prompt must instruct the model to:
   - Answer ONLY based on the provided context
   - Cite which chunk(s) it used
   - Say "I don't have enough information" if the context doesn't contain the answer
6. Test the full pipeline with **5 queries** (3 provided + 2 you create):
   - "How many engineers work in the Seattle office?"
   - "What material was used for the CardioSense enclosure?"
   - "How many hours of continuing education are required annually?"
   - *(your query 1)*
   - *(your query 2)*

### Key concepts

- A vector database (ChromaDB) replaces manual cosine similarity — it indexes
  embeddings and returns nearest neighbors efficiently
- **Metadata** (source document, chunk index) is critical — you need to know WHERE
  the answer came from, not just what it says
- The **generation prompt** matters: without "answer only from context," the model
  will hallucinate from training data instead of your documents
- This is what Claude's web search, Copilot's codebase search, and enterprise RAG
  platforms do at scale

### Reference links

- [ChromaDB Getting Started](https://docs.trychroma.com/docs/overview/getting-started)
- [GitHub Models Quickstart](https://docs.github.com/en/github-models/quickstart)

### What to submit [10 pts]

```
ChromaDB collection created: {N} chunks indexed
```

For each of 5 queries, print:
```
Question: {question}
Retrieved chunks:
  1. [{source_doc_id}] {first 100 chars of chunk}...
  2. [{source_doc_id}] {first 100 chars of chunk}...
  3. [{source_doc_id}] {first 100 chars of chunk}...
Generated answer: {answer from GitHub Models}
Answer correct: [yes/no]. Ground truth: {expected answer}
```

```
RAG pipeline accuracy: {X}/5 queries answered correctly
```

In [ ]:
# --- GitHub PAT Setup ---
# Loads your token from a .env file in your repo root.
# If you don't have a .env file yet, ask Copilot to create one!
# Your .env should contain:   GITHUB_TOKEN=github_pat_your_token_here

import os

# Look for .env in repo root (handles running from notebook dir or repo root)
for _env_dir in [os.path.abspath(""), os.path.abspath(".."), os.path.abspath("../..") ]:
    _env_path = os.path.join(_env_dir, ".env")
    if os.path.exists(_env_path):
        with open(_env_path) as _f:
            for _line in _f:
                _line = _line.strip()
                if _line and not _line.startswith("#") and "=" in _line:
                    _key, _val = _line.split("=", 1)
                    os.environ[_key.strip()] = _val.strip()
        break

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
if not GITHUB_TOKEN:
    print("⚠️  GITHUB_TOKEN not found. The generation cells in Section 4 will fail.")
    print()
    print("To fix this:")
    print("  1. Create a .env file in your repo root (ask Copilot if you need help)")
    print("  2. Add this line:  GITHUB_TOKEN=github_pat_your_token_here")
    print("  3. Re-run this cell")
else:
    print(f"✅ GitHub token loaded ({len(GITHUB_TOKEN)} chars)")

# Helper: API call with retry logic for rate limits
from openai import OpenAI

github_client = OpenAI(
    base_url="https://models.github.ai/inference",
    api_key=GITHUB_TOKEN,
)

def call_github_model(messages, model="openai/gpt-4o", max_retries=3):
    """Call GitHub Models API with retry logic for rate limits."""
    for attempt in range(max_retries):
        try:
            response = github_client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=0.3,
                max_tokens=500,
            )
            return response.choices[0].message.content
        except Exception as e:
            if "rate" in str(e).lower() and attempt < max_retries - 1:
                wait = 15 * (attempt + 1)
                print(f"  Rate limited. Waiting {wait}s... (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
            else:
                raise
    return None

print("GitHub Models client ready.")
print("Note: Free tier rate limits apply (~10 req/min, ~50 req/day for GPT-4o).")
print("If rate-limited, wait or switch model: e.g., 'openai/gpt-4o-mini'")

In [ ]:
# Section 4, Steps 1-3: Chunk documents and create ChromaDB collection

# Chunk size choice: 500 characters with 100-char overlap
# Based on Section 3 analysis, 500 chars balances specificity and context —
# small enough to target individual facts, large enough to preserve surrounding detail.
CHUNK_SIZE = 500
OVERLAP = 100

# Chunk all 20 documents
all_chunks = []
chunk_metadata_ids = []
chunk_metadata_sources = []
chunk_metadata_titles = []

for doc in documents:
    doc_chunks = chunk_text(doc["text"], CHUNK_SIZE, OVERLAP)
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append(chunk)
        chunk_metadata_ids.append(f"{doc['id']}_chunk_{i}")
        chunk_metadata_sources.append(doc["id"])
        chunk_metadata_titles.append(doc["title"])

print(f"Total chunks created: {len(all_chunks)}")

# Embed all chunks
chunk_embeddings = embedding_model.encode(all_chunks, show_progress_bar=True)
print(f"Embeddings shape: {chunk_embeddings.shape}")

# Create ChromaDB collection
chroma_client = chromadb.Client()

# Delete collection if it already exists (for re-runs)
try:
    chroma_client.delete_collection("ridgeline_docs")
except:
    pass

collection = chroma_client.create_collection(
    name="ridgeline_docs",
    metadata={"hnsw:space": "cosine"}
)

# Add all chunks with embeddings and metadata
collection.add(
    ids=chunk_metadata_ids,
    embeddings=chunk_embeddings.tolist(),
    documents=all_chunks,
    metadatas=[
        {"source_doc_id": src, "title": title, "chunk_index": i}
        for src, title, i in zip(
            chunk_metadata_sources,
            chunk_metadata_titles,
            range(len(all_chunks))
        )
    ]
)

print(f"\nChromaDB collection created: {collection.count()} chunks indexed")

In [ ]:
# Section 4, Step 4: query_rag function

def query_rag(question, n_results=3):
    """Embed question, query ChromaDB, return top-N chunks with metadata."""
    # Embed the question
    query_embedding = embedding_model.encode([question]).tolist()

    # Query ChromaDB for nearest neighbors
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    # Package results for downstream use
    retrieved = []
    for i in range(len(results["documents"][0])):
        retrieved.append({
            "chunk_text": results["documents"][0][i],
            "source_doc_id": results["metadatas"][0][i]["source_doc_id"],
            "title": results["metadatas"][0][i]["title"],
            "distance": results["distances"][0][i],
        })

    return retrieved

# Quick test
test_results = query_rag("What is the standard billing rate for senior engineers?")
print("query_rag test — top 3 results:")
for i, r in enumerate(test_results, 1):
    print(f"  {i}. [{r['source_doc_id']}] {r['chunk_text'][:100]}... (distance: {r['distance']:.4f})")

In [ ]:
# Section 4, Step 5: generate_answer function

def generate_answer(question, retrieved_chunks):
    """Format chunks into a prompt, call GitHub Models API, return answer."""
    # Build context from retrieved chunks
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        context_parts.append(
            f"[Chunk {i} — from {chunk['source_doc_id']}: {chunk['title']}]\n"
            f"{chunk['chunk_text']}"
        )
    context = "\n\n".join(context_parts)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant that answers questions ONLY based on "
                "the provided context documents. Follow these rules strictly:\n"
                "1. Answer ONLY using information found in the context below.\n"
                "2. Cite which chunk(s) you used (e.g., [Chunk 1]).\n"
                "3. If the context does not contain enough information to answer, "
                "say: \"I don't have enough information in the provided context.\"\n"
                "4. Keep your answer concise (2-3 sentences max)."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}",
        },
    ]

    answer = call_github_model(messages)
    return answer

# Quick test
test_chunks = query_rag("What is the standard billing rate for senior engineers?")
test_answer = generate_answer("What is the standard billing rate for senior engineers?", test_chunks)
print(f"Test answer: {test_answer}")

In [ ]:
# Section 4, Step 6: Test the full RAG pipeline with 5 queries

queries = [
    {
        "question": "How many engineers work in the Seattle office?",
        "ground_truth": "47 engineers"
    },
    {
        "question": "What material was used for the CardioSense enclosure?",
        "ground_truth": "Glass-filled nylon PA66-GF30"
    },
    {
        "question": "How many hours of continuing education are required annually?",
        "ground_truth": "24 hours per year"
    },
    {
        "question": "What is the tuition reimbursement limit per year?",
        "ground_truth": "$8,500 per calendar year"
    },
    {
        "question": "What was the budget for the CardioSense project?",
        "ground_truth": "$310,000 (actual: $298,500)"
    },
]

correct_count = 0

for q in queries:
    question = q["question"]
    ground_truth = q["ground_truth"]

    # Retrieve
    retrieved = query_rag(question, n_results=3)

    # Generate
    answer = generate_answer(question, retrieved)

    # Print formatted output
    print(f"Question: {question}")
    print("Retrieved chunks:")
    for i, r in enumerate(retrieved, 1):
        print(f"  {i}. [{r['source_doc_id']}] {r['chunk_text'][:100]}...")
    print(f"Generated answer: {answer}")

    # Assess correctness — check if ground truth keywords appear in the answer
    gt_lower = ground_truth.lower()
    answer_lower = answer.lower() if answer else ""
    # Extract key numbers/terms from ground truth for matching
    is_correct = any(term in answer_lower for term in gt_lower.split())
    correct_label = "yes" if is_correct else "no"
    if is_correct:
        correct_count += 1
    print(f"Answer correct: [{correct_label}]. Ground truth: {ground_truth}")
    print("-" * 80)
    time.sleep(2)  # Respect rate limits

print(f"\nRAG pipeline accuracy: {correct_count}/5 queries answered correctly")

---
## Section 5: Capstone — Querying "Attention Is All You Need" [8 pts]

Now apply your RAG pipeline to a real research paper. You'll embed the paper that
invented the Transformer architecture — the foundation of every LLM you've used this
quarter — and query it using the retrieval technique it describes.

The paper text is loaded in `attention_paper` from the setup cell (~30,000 characters
of key sections: Abstract, Introduction, Model Architecture, Attention, Results,
Conclusion). This is an abridged educational summary — read the full paper at
https://arxiv.org/abs/1706.03762

### What to build

1. Chunk the paper using the chunk size you chose in Section 4
2. Add the chunks to a **new** ChromaDB collection (or clear and rebuild the existing
   one) — keep the paper separate from the Ridgeline documents
3. Query with these 4 specific questions:
   - "What problem does multi-head attention solve compared to single attention?"
   - "How does the model handle the order of words without recurrence?"
   - "What were the BLEU scores on the English-to-German translation task?"
   - "Why is self-attention faster than recurrence for sequence modeling?"
4. For each query, print the top-2 retrieved chunks (first 150 chars each) and write
   a 1–2 sentence answer **based on what the chunks say** (not from memory)

### Key concepts

- You're using retrieval-augmented generation to study the paper that invented the
  attention mechanism *behind* retrieval-augmented generation. Recursive and satisfying.
- The paper's technical content is dense — your RAG pipeline should surface the
  specific relevant passages rather than requiring you to read all 15 pages
- Could you answer these by pasting the entire paper into an LLM? Yes — it's short
  enough. But a 500-page engineering manual wouldn't fit. **RAG scales where
  paste-everything doesn't.**

### What to submit [8 pts]

For each of the 4 queries, print:
```
Question: {question}
Chunk 1: {first 150 chars}...
Chunk 2: {first 150 chars}...
Answer: {your 1-2 sentence answer based on the retrieved chunks}
```

Final reflection (print as text in the last cell):
```
One thing RAG got right: {your observation}
One limitation I noticed: {your observation}
```

In [ ]:
# Section 5: Chunk the Attention paper and add to a new ChromaDB collection

# Chunk the paper using the same chunk size from Section 4 (500 chars, 100 overlap)
paper_chunks = chunk_text(attention_paper, CHUNK_SIZE, OVERLAP)
print(f"Attention paper: {len(attention_paper)} characters → {len(paper_chunks)} chunks")

# Embed all paper chunks
paper_embeddings = embedding_model.encode(paper_chunks, show_progress_bar=True)
print(f"Paper embeddings shape: {paper_embeddings.shape}")

# Create a new ChromaDB collection for the paper
try:
    chroma_client.delete_collection("attention_paper")
except:
    pass

paper_collection = chroma_client.create_collection(
    name="attention_paper",
    metadata={"hnsw:space": "cosine"}
)

paper_collection.add(
    ids=[f"paper_chunk_{i}" for i in range(len(paper_chunks))],
    embeddings=paper_embeddings.tolist(),
    documents=paper_chunks,
    metadatas=[
        {"source": "Attention Is All You Need", "chunk_index": i}
        for i in range(len(paper_chunks))
    ]
)

print(f"\nChromaDB collection 'attention_paper' created: {paper_collection.count()} chunks indexed")

In [ ]:
# Section 5: Query the Attention paper with 4 questions

paper_questions = [
    "What problem does multi-head attention solve compared to single attention?",
    "How does the model handle the order of words without recurrence?",
    "What were the BLEU scores on the English-to-German translation task?",
    "Why is self-attention faster than recurrence for sequence modeling?",
]

for question in paper_questions:
    # Query the paper collection
    query_embedding = embedding_model.encode([question]).tolist()
    results = paper_collection.query(
        query_embeddings=query_embedding,
        n_results=2,
        include=["documents", "metadatas", "distances"]
    )

    # Build retrieved chunks for answer generation
    retrieved_chunks = []
    for i in range(len(results["documents"][0])):
        retrieved_chunks.append({
            "chunk_text": results["documents"][0][i],
            "source_doc_id": "Attention Paper",
            "title": "Attention Is All You Need",
            "distance": results["distances"][0][i],
        })

    print(f"Question: {question}")
    for i, chunk in enumerate(retrieved_chunks, 1):
        print(f"Chunk {i}: {chunk['chunk_text'][:150]}...")

    # Generate answer using the GitHub Models API
    answer = generate_answer(question, retrieved_chunks)
    print(f"Answer: {answer}")
    print("-" * 80)
    time.sleep(2)  # Respect rate limits

In [ ]:
# Capstone reflection

print("One thing RAG got right: RAG consistently retrieved the most relevant chunks "
      "for factual questions with specific keywords (like BLEU scores and positional "
      "encoding), surfacing exact passages that contained the answer without needing "
      "to read the entire paper.")

print()

print("One limitation I noticed: For broader conceptual questions, the retrieved chunks "
      "sometimes missed important context that spanned across multiple sections of the "
      "paper — the fixed chunk boundaries can split a complete explanation across two "
      "chunks, leaving either one incomplete on its own.")

---
## Section 6: Reflections [8 pts]

Answer each prompt in 2–3 sentences. Generic responses earn minimal credit — reference
specific results from your work above.

### Reflection 1: How does chunk size affect retrieval quality?

Describe the tradeoff you observed in Section 3. What would you consider when choosing
a chunk size for a new document set?

**Your answer (2–3 sentences):**

*[Replace this with your reflection]*

### Reflection 2: Where did RAG succeed and where did it fail?

Across Sections 4 and 5, identify one query where RAG found exactly the right
information and one where it struggled. Why?

**Your answer (2–3 sentences):**

*[Replace this with your reflection]*

### Reflection 3: How will you apply context management in Part B?

You'll be building a context package for the MiniClaw project in Part B. Based on
what you learned about chunking, retrieval, and the information problem, what's your
strategy?

**Your answer (2–3 sentences):**

*[Replace this with your reflection]*

---
## Submission Checklist

Before submitting, verify:

- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Section 1: Token counts and context budget printed
- [ ] Section 2: Ranked document lists for both queries with scores
- [ ] Section 3: Comparison table across 3 chunk sizes, top chunks printed
- [ ] Section 4: ChromaDB collection created, 5 queries with generated answers and accuracy
- [ ] Section 5: 4 paper queries answered with retrieved evidence + reflection
- [ ] Section 6: Three reflections filled in (2–3 sentences each, specific to your results)

**Due: Monday, April 27, 2026 at 11:59 PM**

**Save and push your work:**
In Codespaces, use the Source Control panel (Ctrl+Shift+G) to commit and push.
Then paste your GitHub repo URL in Canvas.

---

### References

- Vaswani, A. et al. "Attention Is All You Need." NeurIPS, 2017. https://arxiv.org/abs/1706.03762
- ChromaDB documentation: https://docs.trychroma.com/docs/overview/introduction
- GitHub Models: https://docs.github.com/en/github-models
- Sentence-Transformers: https://www.sbert.net/
- tiktoken: https://github.com/openai/tiktoken